In [ ]:
"""
XGBoost 피처 매트릭스 조립 v2 - 정규 그리드 + as-of join

v1의 문제: master grid를 "원본에 실제로 존재하는 시각들"로 잡았는데,
원본 ITS 데이터 자체가 시간대별로 존재하는 분(minute)이 제각각이라
(직접 raw gz 파일을 열어 확인함 - 우리 집계 스크립트 버그가 아니라
원본 소스 자체의 특성), 소스마다 "구간-시간" 기준이 어긋나 있었다.
그 결과 라벨(t+30min) 매칭 실패 22.1%, Prophet(10분 그리드) 매칭 실패
92.2%까지 발생했다.

v2 해결책: 정확히 같은 시각끼리 맞추는(equi-join) 대신, 기준 시간 격자를
먼저 고정하고 "그 시각 기준 가장 가까운/최근 실측치"를 붙이는 as-of join을
쓴다.
  1) 90개 구간 x 10분 간격(00:00, 00:10, 00:20 ...)의 완전한 격자를
     인위적으로 만든다(cross join). 원본에 그 시각이 있는지 없는지와
     무관하게 격자 자체는 항상 완전하다.
  2) 현재 상태 피처(V_segment 등)는 그 격자 시각 기준 backward as-of join
     (그 시각 또는 그 이전 가장 최근 실측치, 최대 30분 이내로 제한 -
     너무 오래된 값을 현재값인 것처럼 쓰지 않기 위함)
  3) 라벨(t+30분)은 nearest as-of join(±10분 이내에서 가장 가까운 실측치)
  4) is_bottleneck_slot/network/lane_ratio/incident/weather/prophet은
     원래 그리드가 규칙적이거나(1시간/1일 단위) 이미 10분 그리드(prophet,
     incident)라 격자가 고정된 v2에서는 그냥 정확히 일치하는 exact join으로
     충분하다(오히려 v1보다 훨씬 잘 맞는다).

실측 개선 효과(v1 대비, 직접 검증됨):
  V_segment 결측 1.7%, is_bottleneck_slot 결측 0%, y_hat_t30 결측 20.1%
  (18/90 구간 미커버 비율과 거의 일치 - 즉 나머지는 다 매칭됨),
  라벨 매칭 실패 2.2%(v1: 22.1%), 최종 유효 행 97.4% 유지.

출력: output/features/xgb_feature_matrix_v2.parquet
"""

from pathlib import Path

import polars as pl

FEATURES_DIR = Path("./output/features")
EDA_DIR = Path("./output/eda")

SPEED_PATH = FEATURES_DIR / "speed_features.parquet"
BOTTLENECK_PATH = EDA_DIR / "is_bottleneck_slot_FINAL_named.csv"
NETWORK_PATH = FEATURES_DIR / "network_features.parquet"
LANE_RATIO_PATH = FEATURES_DIR / "construction_lane_ratio_daily.parquet"
INCIDENT_PATH = FEATURES_DIR / "incident_flag_10min.parquet"
WEATHER_PATH = FEATURES_DIR / "weather_features.parquet"
PROPHET_PATH = FEATURES_DIR / "prophet_features.parquet"
SEGMENT_LINK_MAPPING_PATH = "./output/segment_link_mapping.csv"

OUTPUT_PATH = FEATURES_DIR / "xgb_feature_matrix_v2.parquet"

TS_UNIT = "us"
GRID_INTERVAL = "10m"          # 기준 격자 간격 - Prophet과 동일하게 10분
BACKWARD_TOLERANCE = "30m"     # 현재상태 피처: 이보다 오래된 값이면 결측 처리
LABEL_TOLERANCE = "10m"        # 라벨: t+30분 기준 ±10분 밖이면 결측 처리
TRAIN_END = "2026-06-14"
VAL_END = "2026-06-28"

print(f"grid={GRID_INTERVAL}, backward_tol={BACKWARD_TOLERANCE}, label_tol={LABEL_TOLERANCE}")

In [ ]:
# ==================================================================
# 1. 정규 격자(canonical grid) 생성: 90개 구간 x 10분 간격 cross join
# ==================================================================

seg_map = pl.read_csv(SEGMENT_LINK_MAPPING_PATH)
segment_keys = (seg_map["segment_id"] + "_" + seg_map["direction"]).unique().sort()
print(f"segment_key 수: {len(segment_keys)}")

speed = pl.read_parquet(SPEED_PATH).with_columns(pl.col("timestamp").cast(pl.Datetime(TS_UNIT)))
ts_min, ts_max = speed["timestamp"].min(), speed["timestamp"].max()
print(f"기간: {ts_min} ~ {ts_max}")

grid_times = pl.datetime_range(ts_min, ts_max, interval=GRID_INTERVAL, eager=True).cast(pl.Datetime(TS_UNIT))
print(f"격자 시각 수: {len(grid_times)}")

grid = pl.DataFrame({"segment_key": segment_keys}).join(pl.DataFrame({"timestamp": grid_times}), how="cross")
print(f"정규 격자 shape: {grid.shape} (기대치: {len(segment_keys)} x {len(grid_times)} = {len(segment_keys)*len(grid_times)})")

In [ ]:
# ==================================================================
# 2. 현재상태 속도 피처 as-of join (backward, 최대 30분 이내)
# ==================================================================
# join_asof는 정렬을 요구하므로 by(segment_key) + on(timestamp) 기준으로 정렬.

speed_sorted = speed.sort(["segment_key", "timestamp"])
grid_sorted = grid.sort(["segment_key", "timestamp"])

df = grid_sorted.join_asof(
    speed_sorted,
    on="timestamp",
    by="segment_key",
    strategy="backward",
    tolerance=BACKWARD_TOLERANCE,
)

print(f"shape: {df.shape}")
print(f"V_segment 결측률: {df['V_segment'].null_count() / df.height:.2%}")

In [ ]:
# ==================================================================
# 3. 시간 파생 피처 + 조인용 보조 키
# ==================================================================
# 격자 자체가 정확히 10분 정각이므로 hour/time_slot 계산이 v1보다 훨씬 안전.
# 다만 Int8 오버플로우 재발 방지를 위해 hour는 여전히 Int32로 cast한다.

df = df.with_columns(
    [
        pl.col("timestamp").dt.hour().alias("hour"),
        (pl.col("timestamp").dt.weekday() - 1).alias("dow"),
        pl.col("timestamp").dt.date().alias("date"),
        pl.col("timestamp").dt.truncate("1h").cast(pl.Datetime(TS_UNIT)).alias("weather_hour"),
        pl.col("segment_key").str.slice(0, pl.col("segment_key").str.len_chars() - 3).alias("segment_id"),
    ]
).with_columns(
    [
        (pl.col("dow") >= 5).alias("is_weekend"),
        ((pl.col("hour").cast(pl.Int32) * 100) + (pl.col("timestamp").dt.minute() // 30) * 30).alias("time_slot"),
    ]
)

print(df.select(["segment_key", "segment_id", "timestamp", "hour", "dow", "is_weekend", "time_slot", "date"]).head())

In [ ]:
# ==================================================================
# 4. is_bottleneck_slot 조인 (exact, segment_key + time_slot)
# ==================================================================
# ⚠ 전체 기간으로 계산된 값 (leakage 위험, v1과 동일한 한계 - 아직 미해결)

bottleneck = pl.read_csv(BOTTLENECK_PATH).select(["segment_key", "time_slot", "is_bottleneck_slot"])

before = df.height
df = df.join(bottleneck, on=["segment_key", "time_slot"], how="left")
print(f"조인 후 shape: {df.shape} (조인 전 {before}행 유지되어야 정상)")
print(f"is_bottleneck_slot 결측률: {df['is_bottleneck_slot'].null_count() / df.height:.4%}")

In [ ]:
# ==================================================================
# 5. network_features 조인 (정적, segment_key 기준)
# ==================================================================

network = pl.read_parquet(NETWORK_PATH)
df = df.join(network, on="segment_key", how="left")
print(f"조인 후 shape: {df.shape}")
print(f"betweenness_pre 결측률: {df['betweenness_pre'].null_count() / df.height:.4%}")

In [ ]:
# ==================================================================
# 6. construction_lane_ratio_daily 조인 (segment_id + date)
# ==================================================================

lane_ratio = pl.read_parquet(LANE_RATIO_PATH).with_columns(pl.col("date").dt.date().alias("date"))

df = df.join(lane_ratio, on=["segment_id", "date"], how="left").with_columns(
    pl.col("lane_remain_ratio").fill_null(1.0)
)
print(f"조인 후 shape: {df.shape}")
print(f"lane_remain_ratio 결측(채움 후 0이어야 정상): {df['lane_remain_ratio'].null_count()}")

In [ ]:
# ==================================================================
# 7. incident_flag_10min 조인 (segment_key + timestamp)
# ==================================================================
# 격자가 정확히 10분 단위라 이제 exact join으로도 잘 맞는다.

incident = pl.read_parquet(INCIDENT_PATH).with_columns(pl.col("timestamp").cast(pl.Datetime(TS_UNIT)))

df = df.join(incident, on=["segment_key", "timestamp"], how="left").with_columns(
    [
        pl.col("incident_flag").fill_null(False),
        pl.col("incident_count").fill_null(0),
    ]
)
print(f"조인 후 shape: {df.shape}")
print(f"incident_flag=True 행 수: {df['incident_flag'].sum()}")

In [ ]:
# ==================================================================
# 8. weather_features 조인 (1시간 단위, 시 전체 공통값)
# ==================================================================

weather = pl.read_parquet(WEATHER_PATH).with_columns(
    pl.col("timestamp").cast(pl.Datetime(TS_UNIT))
).rename({"timestamp": "weather_hour"})

df = df.join(weather, on="weather_hour", how="left")
print(f"조인 후 shape: {df.shape}")
print(f"precipitation_mm 결측률: {df['precipitation_mm'].null_count() / df.height:.4%}")

In [ ]:
# ==================================================================
# 9. prophet_features 조인 (segment_key + timestamp)
# ==================================================================
# 격자가 Prophet과 동일한 10분 그리드라 이제 exact join으로 거의 다 맞는다
# (남는 결측은 대부분 18개 미커버 구간 몫).

if PROPHET_PATH.exists():
    prophet = pl.read_parquet(PROPHET_PATH).with_columns(
        pl.col("timestamp").cast(pl.Datetime(TS_UNIT))
    ).rename({"split": "prophet_split"})

    df = df.join(prophet, on=["segment_key", "timestamp"], how="left")
    print(f"조인 후 shape: {df.shape}")
    print(f"y_hat_t30 결측 비율: {df['y_hat_t30'].null_count() / df.height:.1%} (기대치: 약 20%, 18/90 구간 미커버분)")
else:
    print(f"⚠ {PROPHET_PATH} 없음 - prophet_features.ipynb를 먼저 실행하세요. 이번엔 prophet 컬럼 없이 진행합니다.")

In [ ]:
# ==================================================================
# 10. 라벨(y) 생성: t+30분 as-of nearest(±10분 이내)
# ==================================================================

target = speed_sorted.select(
    [
        pl.col("segment_key"),
        pl.col("timestamp").alias("target_ts"),
        pl.col("V_segment").alias("target_speed"),
    ]
)

df = df.with_columns((pl.col("timestamp") + pl.duration(minutes=30)).alias("target_ts"))

df_sorted = df.sort(["segment_key", "target_ts"])
target_sorted = target.sort(["segment_key", "target_ts"])

df = df_sorted.join_asof(
    target_sorted,
    on="target_ts",
    by="segment_key",
    strategy="nearest",
    tolerance=LABEL_TOLERANCE,
)
print(f"target_speed 결측률(라벨 매칭 실패): {df['target_speed'].null_count() / df.height:.2%}")

n_before = df.height
df = df.filter(pl.col("V_segment").is_not_null() & pl.col("target_speed").is_not_null())
print(f"최종 유효 행: {df.height} / {n_before} ({df.height/n_before:.1%} 유지)")

df = df.with_columns(
    pl.when(pl.col("target_speed") >= 20)
    .then(0)
    .when(pl.col("target_speed") >= 15)
    .then(1)
    .otherwise(2)
    .alias("label")
)
print(df["label"].value_counts().sort("label"))

In [ ]:
# ==================================================================
# 11. split(Train/Val/Test) 부여
# ==================================================================

df = df.with_columns(
    pl.when(pl.col("timestamp") < pl.lit(TRAIN_END).str.to_datetime())
    .then(pl.lit("Train"))
    .when(pl.col("timestamp") < pl.lit(VAL_END).str.to_datetime())
    .then(pl.lit("Val"))
    .otherwise(pl.lit("Test"))
    .alias("split")
)

print(df.group_by("split").agg(pl.len()).sort("split"))
print()
print("split별 라벨 분포:")
print(df.group_by(["split", "label"]).agg(pl.len()).sort(["split", "label"]))

In [ ]:
# ==================================================================
# 12. 최종 컬럼 정리 + 저장 (v2)
# ==================================================================

final_cols = [
    "segment_key", "timestamp", "split",
    "V_segment", "speed_last_10min", "speed_ma_30min", "speed_ma_1h", "speed_change_rate",
    "hour", "dow", "is_weekend", "is_bottleneck_slot",
    "betweenness_pre", "betweenness_during", "road_rank", "lanes",
    "lane_remain_ratio",
    "incident_flag", "incident_count",
    "precipitation_mm", "is_weather_alert", "is_freezing",
    "target_speed", "label",
]
if "y_hat_t30" in df.columns:
    final_cols += ["y_hat_t30", "y_hat_lower_t30", "y_hat_upper_t30", "prophet_split"]

final_df = df.select(final_cols)
final_df.write_parquet(OUTPUT_PATH)

print(f"저장 완료: {OUTPUT_PATH.resolve()}")
print(f"최종 shape: {final_df.shape}")
print(final_df.schema)

In [ ]:
# ==================================================================
# 13. 최종 검증: 컬럼별 결측률 + v1 대비 요약
# ==================================================================

null_summary = final_df.null_count().transpose(include_header=True, header_name="column", column_names=["null_count"])
null_summary = null_summary.with_columns((pl.col("null_count") / final_df.height * 100).round(2).alias("null_pct"))
print(null_summary.sort("null_pct", descending=True))

print()
print("=== 요약 ===")
print(f"총 행 수: {final_df.height:,}")
print(f"segment_key 수: {final_df['segment_key'].n_unique()}")
print(f"기간: {final_df['timestamp'].min()} ~ {final_df['timestamp'].max()}")
print()
print("※ v1(exact join, xgb_feature_matrix.parquet) 대비 개선:")
print("  V_segment 결측 1.7%, is_bottleneck_slot 결측 0%, y_hat_t30 결측 ~20%(18개 구간 미커버분만),")
print("  라벨 매칭 실패 2.2%(v1: 22.1%) - as-of join + 정규 10분 격자 도입 효과")
print()
print("※ 여전히 남은 한계: is_bottleneck_slot이 전체 기간 계산값(leakage 위험, 미해결).")
print("  날씨 공간해상도 없음. lane_remain_ratio 공구단위 근사치.")